# ASIC Pipeline Walkthrough

This notebook mirrors the current ASIC pipeline in two explicit stages:

1. harmonize the ASIC source tables into shared static and dynamic tables,
2. review an overview of the harmonized data and essential QC information,
3. build the Chapter 1 cohort and create the 8-hour block outputs.

The notebook intentionally calls the shared pipeline code from the repository rather than re-implementing the logic locally.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 260)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "icu_data_platform").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/icu_data_platform")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from icu_data_platform.sources.asic.extract.raw_tables import (  # noqa: E402
    DEFAULT_ASIC_RAW_DATA_DIR,
    DEFAULT_TRANSLATION_PATH,
)
from icu_data_platform.sources.asic.pipeline import (  # noqa: E402
    DEFAULT_ASIC_HARMONIZED_OUTPUT_DIR,
    build_asic_chapter1_dataset,
    build_asic_harmonized_dataset,
    write_asic_chapter1_dataset,
    write_asic_harmonized_dataset,
)

PROJECT_ROOT


PosixPath('/Users/joanameyer/repository/icu-data-platform')

## Pipeline Parameters


In [2]:
PIPELINE_PARAMETERS = {
    "raw_dir": DEFAULT_ASIC_RAW_DATA_DIR,
    "translation_path": DEFAULT_TRANSLATION_PATH,
    "min_non_null": 20,
    "min_hospitals": 4,
    "fence_factor": 1.5,
}
OUTPUT_DIR = PROJECT_ROOT / DEFAULT_ASIC_HARMONIZED_OUTPUT_DIR

if not PIPELINE_PARAMETERS["raw_dir"].exists():
    raise FileNotFoundError(f"ASIC raw data directory not found: {PIPELINE_PARAMETERS['raw_dir']}")

display(pd.Series({key: str(value) for key, value in PIPELINE_PARAMETERS.items()}, name="value").to_frame())
print(f"Output directory: {OUTPUT_DIR}")


,value
raw_dir,/Users/joanameyer/data/asic/synthetic_control_sample10
translation_path,/Users/joanameyer/repository/icu-data-platform/src/icu_data_platform/sources/asic/column_translation.json
min_non_null,20
min_hospitals,4
fence_factor,1.5


Output directory: /Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized


## 1. Harmonize ASIC Data


In [3]:
harmonized_dataset = build_asic_harmonized_dataset(**PIPELINE_PARAMETERS)
harmonized_output_paths = write_asic_harmonized_dataset(
    harmonized_dataset,
    output_dir=OUTPUT_DIR,
)

final_static = harmonized_dataset.static.combined.copy()
final_dynamic = harmonized_dataset.dynamic.combined.copy()

harmonized_overview = pd.DataFrame(
    [
        {
            "table": "static",
            "rows": final_static.shape[0],
            "columns": final_static.shape[1],
            "hospitals": final_static["hospital_id"].nunique(dropna=True),
            "unique_stays": final_static["stay_id_global"].nunique(dropna=True),
        },
        {
            "table": "dynamic",
            "rows": final_dynamic.shape[0],
            "columns": final_dynamic.shape[1],
            "hospitals": final_dynamic["hospital_id"].nunique(dropna=True),
            "unique_stays": final_dynamic["stay_id_global"].nunique(dropna=True),
        },
    ]
)

display(Markdown("### Harmonized Output Paths"))
display(pd.Series({key: str(path) for key, path in harmonized_output_paths.items()}, name="path").to_frame())

display(Markdown("### Harmonized Table Overview"))
display(harmonized_overview)


### Harmonized Output Paths

,path
static_harmonized,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/harmonized.csv
static_source_map,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/source_map.csv
static_schema_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/schema_summary.csv
static_categorical_value_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/static/categorical_value_summary.csv
dynamic_harmonized,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/harmonized.csv
dynamic_source_map,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/source_map.csv
dynamic_schema_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/schema_summary.csv
dynamic_non_numeric_issues,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/non_numeric_issues.csv
dynamic_semantic_decisions,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/semantic_decisions.csv
dynamic_invalid_value_rules,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/dynamic/invalid_value_rules.csv


### Harmonized Table Overview

,table,rows,columns,hospitals,unique_stays
0,static,80,16,8,80
1,dynamic,105165,107,8,80


### Harmonized Data Overview and Essential Information


In [4]:
stay_id_qc_summary = harmonized_dataset.stay_id_qc.summary.copy()

static_rows_by_hospital = (
    final_static.groupby("hospital_id", dropna=False)
    .size()
    .rename("static_rows")
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

dynamic_time_coverage = (
    final_dynamic.assign(parsed_time=pd.to_timedelta(final_dynamic["time"], errors="coerce"))
    .groupby("hospital_id", dropna=False)
    .agg(
        dynamic_rows=("time", "size"),
        unique_stays=("stay_id_global", lambda s: s.nunique(dropna=True)),
        min_time=("parsed_time", "min"),
        max_time=("parsed_time", "max"),
    )
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)

qc_snapshot = pd.DataFrame(
    [
        {
            "artifact": "stay_id_mapping_failures",
            "rows": harmonized_dataset.stay_id_qc.mapping_failures.shape[0],
        },
        {
            "artifact": "duplicate_dynamic_time_index",
            "rows": harmonized_dataset.stay_id_qc.duplicate_dynamic_time_index.shape[0],
        },
        {
            "artifact": "dynamic_non_numeric_issues",
            "rows": harmonized_dataset.dynamic.non_numeric_issues.shape[0],
        },
        {
            "artifact": "dynamic_distribution_issues",
            "rows": harmonized_dataset.dynamic.distribution_issues.shape[0],
        },
    ]
)

static_preview_columns = [
    column
    for column in ["hospital_id", "stay_id_global", "stay_id_local", "sex", "age_group", "readmission", "icu_mortality"]
    if column in final_static.columns
]
dynamic_preview_columns = [
    column
    for column in ["hospital_id", "stay_id_global", "time", "minutes_since_admit", "heart_rate", "map", "sbp", "dbp", "spo2", "resp_rate"]
    if column in final_dynamic.columns
]

display(Markdown("### Stay-ID QC Summary"))
display(stay_id_qc_summary)

display(Markdown("### Static Rows by Hospital"))
display(static_rows_by_hospital)

display(Markdown("### Dynamic Time Coverage by Hospital"))
display(dynamic_time_coverage)

display(Markdown("### QC Snapshot"))
display(qc_snapshot)

display(Markdown("### Static Preview"))
display(final_static[static_preview_columns].head(10))

display(Markdown("### Dynamic Preview"))
display(final_dynamic[dynamic_preview_columns].head(10))

display(Markdown("### Static Schema Summary"))
display(harmonized_dataset.static.schema_summary)

display(Markdown("### Dynamic Schema Summary"))
display(harmonized_dataset.dynamic.schema_summary)


### Stay-ID QC Summary

,metric,value
0,static_rows_total,80
1,static_unique_stay_id_global_total,80
2,unique_stays_hospital_count,8
3,duplicated_stay_id_local_values_across_hospitals,10
4,failed_or_missing_static_dynamic_mappings,0
5,rows_missing_hospital_id_or_stay_id_local,0
6,duplicate_static_stay_id_global_rows,0
7,duplicate_dynamic_rows_within_stay_time,0


### Static Rows by Hospital

,hospital_id,static_rows
0,asic_UK00,10
1,asic_UK01,10
2,asic_UK02,10
3,asic_UK03,10
4,asic_UK04,10
5,asic_UK06,10
6,asic_UK07,10
7,asic_UK08,10


### Dynamic Time Coverage by Hospital

...

,hospital_id,stay_id_global,time,minutes_since_admit,heart_rate,map,sbp,dbp,spo2,resp_rate
0,asic_UK00,asic_UK00_9990,0 days 00:00:00,0.0,102.0,71.0,142.0,64.0,94.0,NaN
1,asic_UK00,asic_UK00_9990,0 days 00:15:00,15.0,98.0,80.0,117.0,99.0,95.0,NaN
2,asic_UK00,asic_UK00_9990,0 days 00:30:00,30.0,95.0,65.0,123.0,45.0,95.0,NaN
3,asic_UK00,asic_UK00_9990,0 days 00:45:00,45.0,90.0,64.0,139.0,86.0,96.0,NaN
4,asic_UK00,asic_UK00_9990,0 days 01:00:00,60.0,60.0,66.5,137.0,77.0,92.0,NaN
5,asic_UK00,asic_UK00_9990,0 days 01:15:00,75.0,86.0,63.0,110.0,53.0,100.0,NaN
6,asic_UK00,asic_UK00_9990,0 days 01:30:00,90.0,75.0,79.0,115.0,52.0,99.0,NaN
7,asic_UK00,asic_UK00_9990,0 days 01:45:00,105.0,98.0,79.0,113.0,48.5,96.0,NaN
8,asic_UK00,asic_UK00_9990,0 days 02:00:00,120.0,82.0,63.0,101.0,51.0,95.0,NaN
9,asic_UK00,asic_UK00_9990,0 days 02:15:00,135.0,72.0,68.0,102.0,62.0,94.0,NaN


### Static Schema Summary

,hospital,rows,final_columns,empty_columns
0,asic_UK00,10,16,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
1,asic_UK01,10,16,"[bmi_group, hosp_mortality, icu_mortality, hosp_los, icu_los, icu_readmit, dialysis_free_days, vent_free_days]"
2,asic_UK02,10,16,[]
3,asic_UK03,10,16,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
4,asic_UK04,10,16,"[dialysis_free_days, vent_free_days]"
5,asic_UK06,10,16,[]
6,asic_UK07,10,16,[]
7,asic_UK08,10,16,"[hosp_los, dialysis_free_days, vent_free_days]"


### Dynamic Schema Summary

,hospital,rows,final_columns,empty_columns
0,asic_UK00,11565,107,"[feo2, cardiac_output_bolus, cardiac_index_bolus, stroke_volume_bolus, stroke_index_bolus, pvri, gedvi, evlwi, ntprobnp, ecmo_o2, extracorp_blood_flow, extracorp_o2_flow, inhaled_iloprost, dobutam..."
1,asic_UK01,6083,107,"[heart_rate, sbp, map, dbp, resp_rate, spont_resp_rate, core_temp, spo2, sao2, scvo2, cvp, fio2, feo2, compliance, ie_ratio, vt, etco2, pao2, paco2, ph_art, bicarbonate_art, base_excess_art, lacta..."
2,asic_UK02,12116,107,"[spont_resp_rate, core_temp, sao2, cvp, feo2, delta_p, vt, ph_art, pap_sys, pap_mean, pap_dias, pcwp, cardiac_output_bolus, cardiac_output_cont, cardiac_index_bolus, cardiac_index_cont, stroke_vol..."
3,asic_UK03,19201,107,"[sbp, map, dbp, scvo2, feo2, ie_ratio, pap_sys, pap_mean, pap_dias, cardiac_output_cont, cardiac_index_bolus, cardiac_index_cont, stroke_volume_bolus, stroke_volume_cont, stroke_index_bolus, strok..."
4,asic_UK04,9025,107,"[scvo2, feo2, vt, pap_sys, pap_mean, pap_dias, pcwp, cardiac_output_cont, stroke_volume_bolus, stroke_index_bolus, pvri, d_dimer, bnp, ntprobnp, fluid_balance_24h, position_therapy, ecmo, ecmo_o2,..."
5,asic_UK06,11677,107,"[heart_rate, sbp, map, dbp, resp_rate, spont_resp_rate, core_temp, spo2, sao2, scvo2, cvp, fio2, feo2, delta_p, compliance, ie_ratio, vt, vt_per_kg_ibw, etco2, pao2, paco2, ph_art, bicarbonate_art..."
6,asic_UK07,19170,107,"[feo2, compliance, cardiac_output_bolus, cardiac_output_cont, stroke_volume_bolus, stroke_volume_cont, stroke_index_bolus, stroke_index_cont, pvri, bnp, ck_mb, extracorp_blood_flow, inhaled_no, in..."
7,asic_UK08,16328,107,"[scvo2, cvp, feo2, vt, etco2, pcwp, cardiac_output_bolus, cardiac_output_cont, cardiac_index_bolus, cardiac_index_cont, stroke_volume_bolus, stroke_volume_cont, stroke_index_bolus, stroke_index_co..."


## 3. Build Chapter 1 Cohort and 8-Hour Blocks


In [5]:
chapter1_dataset = build_asic_chapter1_dataset(
    harmonized_dataset,
    raw_dir=PIPELINE_PARAMETERS["raw_dir"],
)
chapter1_output_paths = write_asic_chapter1_dataset(
    chapter1_dataset,
    output_dir=OUTPUT_DIR,
)

final_cohort = chapter1_dataset.cohort.table.copy()
chapter1_stays = chapter1_dataset.cohort.chapter1.table.copy()
chapter1_block_index = chapter1_dataset.chapter1_8h_blocks.block_index.copy()
chapter1_blocked_dynamic_features = chapter1_dataset.chapter1_8h_blocks.blocked_dynamic_features.copy()

chapter1_overview = pd.DataFrame(
    [
        {
            "artifact": "authoritative_stay_level_cohort",
            "rows": final_cohort.shape[0],
            "columns": final_cohort.shape[1],
        },
        {
            "artifact": "chapter1_retained_stays",
            "rows": chapter1_stays.shape[0],
            "columns": chapter1_stays.shape[1],
        },
        {
            "artifact": "chapter1_8h_block_index",
            "rows": chapter1_block_index.shape[0],
            "columns": chapter1_block_index.shape[1],
        },
        {
            "artifact": "chapter1_8h_blocked_dynamic_features",
            "rows": chapter1_blocked_dynamic_features.shape[0],
            "columns": chapter1_blocked_dynamic_features.shape[1],
        },
    ]
)

display(Markdown("### Chapter 1 Output Paths"))
display(pd.Series({key: str(path) for key, path in chapter1_output_paths.items()}, name="path").to_frame())

display(Markdown("### Chapter 1 Overview"))
display(chapter1_overview)

display(Markdown("### Retained Hospitals"))
display(chapter1_dataset.cohort.chapter1.retained_hospitals)

display(Markdown("### Chapter 1 Counts by Hospital"))
display(chapter1_dataset.cohort.chapter1.counts_by_hospital)

display(Markdown("### Block QC Summary"))
display(chapter1_dataset.chapter1_8h_blocks.qc_summary)

display(Markdown("### Block Index Preview"))
display(chapter1_block_index.head(10))

display(Markdown("### Blocked Dynamic Feature Preview"))
blocked_preview_columns = [
    column
    for column in [
        "stay_id_global",
        "hospital_id",
        "block_index",
        "block_start_h",
        "block_end_h",
        "dynamic_row_count",
        "non_missing_measurements_in_block",
        "observed_variables_in_block",
        "heart_rate_obs_count",
        "heart_rate_median",
        "heart_rate_last",
        "map_obs_count",
        "map_median",
        "map_last",
        "sbp_obs_count",
        "sbp_median",
        "sbp_last",
        "spo2_obs_count",
        "spo2_median",
        "spo2_last",
    ]
    if column in chapter1_blocked_dynamic_features.columns
]
display(chapter1_blocked_dynamic_features[blocked_preview_columns].head(10))


### Chapter 1 Output Paths

,path
cohort_stay_level,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/stay_level.csv
cohort_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/summary.csv
cohort_preprocessing_notes,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/preprocessing_notes.csv
cohort_icu_end_time_proxy_summary_by_hospital,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/icu_end_time_proxy_summary_by_hospital.csv
cohort_coding_distribution_by_hospital,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/coding_distribution_by_hospital.csv
cohort_chapter1_stay_level,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/chapter1_stay_level.csv
cohort_chapter1_notes,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/chapter1_notes.csv
cohort_chapter1_core_vital_group_coverage,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/chapter1_core_vital_group_coverage.csv
cohort_chapter1_site_eligibility,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/chapter1_site_eligibility.csv
cohort_chapter1_site_counts_summary,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/cohort/chapter1_site_counts_summary.csv


### Chapter 1 Overview

,artifact,rows,columns
0,authoritative_stay_level_cohort,80,7
1,chapter1_retained_stays,39,12
2,chapter1_8h_block_index,1724,6
3,chapter1_8h_blocked_dynamic_features,1724,621


### Retained Hospitals

...

,metric,value,note
0,block_size_hours,8,"Blocks are fixed, non-overlapping, and half-open: [0, 8), [8, 16), [16, 24), ..."
1,retained_chapter1_stays_total,39,Retained ASIC Chapter 1 stays entering 8-hour block construction.
2,retained_dynamic_rows_total,55785,Harmonized dynamic rows after Chapter 1 site-level and stay-level exclusions.
3,retained_stays_with_zero_blocks,0,Retained stays with no fully completed 8-hour block because icu_end_time_proxy < 8h or is unusable.
4,retained_stays_with_one_or_more_blocks,39,Retained stays with at least one fully completed 8-hour block.
5,block_rows_total,1724,Block-level index rows emitted for retained ASIC Chapter 1 stays.
6,negative_dynamic_time_rows_in_retained_input,0,Invalid retained-input dynamic rows with time < 0 hours.
7,negative_dynamic_time_stays_in_retained_input,0,Retained stays containing one or more invalid negative dynamic times.
8,all_constructed_blocks_end_on_or_before_icu_end_time_proxy,1,Confirms that only fully completed blocks with block_end_h <= icu_end_time_proxy were constructed.
9,retained_stays_ending_exactly_on_8h_boundary,1,Retained stays whose icu_end_time_proxy lands exactly on an 8-hour boundary.


### Block Index Preview

,stay_id_global,hospital_id,block_index,block_start_h,block_end_h,prediction_time_h
0,asic_UK02_9990,asic_UK02,0,0,8,8
1,asic_UK02_9990,asic_UK02,1,8,16,16
2,asic_UK02_9990,asic_UK02,2,16,24,24
3,asic_UK02_9990,asic_UK02,3,24,32,32
4,asic_UK02_9990,asic_UK02,4,32,40,40
5,asic_UK02_9990,asic_UK02,5,40,48,48
6,asic_UK02_9990,asic_UK02,6,48,56,56
7,asic_UK02_9990,asic_UK02,7,56,64,64
8,asic_UK02_9990,asic_UK02,8,64,72,72
9,asic_UK02_9990,asic_UK02,9,72,80,80


### Blocked Dynamic Feature Preview

,stay_id_global,hospital_id,block_index,block_start_h,block_end_h,dynamic_row_count,non_missing_measurements_in_block,observed_variables_in_block,heart_rate_obs_count,heart_rate_median,heart_rate_last,map_obs_count,map_median,map_last,sbp_obs_count,sbp_median,sbp_last,spo2_obs_count,spo2_median,spo2_last
0,asic_UK02_9990,asic_UK02,0,0,8,31,238,35,6,87.5,87.0,6,76.5,82.0,6,123.0,133.0,6,100.0,100.0
1,asic_UK02_9990,asic_UK02,1,8,16,32,245,35,4,107.5,117.0,4,77.0,98.0,4,99.5,98.0,1,98.0,98.0
2,asic_UK02_9990,asic_UK02,2,16,24,32,205,25,3,106.0,130.0,3,84.0,77.0,3,133.0,107.0,1,95.0,95.0
3,asic_UK02_9990,asic_UK02,3,24,32,32,274,28,4,94.5,103.0,4,79.0,72.0,3,151.0,159.0,0,NaN,NaN
4,asic_UK02_9990,asic_UK02,4,32,40,32,248,34,5,100.0,84.0,5,67.0,65.0,5,119.0,107.0,0,NaN,NaN
5,asic_UK02_9990,asic_UK02,5,40,48,32,232,25,4,100.5,153.0,4,78.0,70.0,4,112.5,125.0,2,100.0,100.0
6,asic_UK02_9990,asic_UK02,6,48,56,32,229,25,4,78.5,87.0,4,82.5,86.0,4,118.0,133.0,4,97.0,95.0
7,asic_UK02_9990,asic_UK02,7,56,64,32,251,35,5,83.0,48.0,5,76.0,71.0,5,116.0,116.0,5,98.0,100.0
8,asic_UK02_9990,asic_UK02,8,64,72,32,228,24,4,82.0,64.0,4,71.0,73.0,4,117.5,122.0,4,100.0,48.0
9,asic_UK02_9990,asic_UK02,9,72,80,32,220,24,4,93.0,93.0,4,69.5,91.0,4,144.0,75.0,4,94.0,93.0


In [6]:
chapter1_blocked_dynamic_features

# Find all _median features in the blocked dynamic features table
median_features = [column for column in chapter1_blocked_dynamic_features.columns if "_median" in column]
median_df = chapter1_blocked_dynamic_features[["stay_id_global", "hospital_id"] + median_features]
# Find the missingness rate per hospital for each median feature
median_missingness = (
    median_df.groupby("hospital_id", dropna=False)
    .agg({feature: lambda s: s.isna().mean() for feature in median_features})
    .reset_index()
    .sort_values("hospital_id")
    .reset_index(drop=True)
)
median_missingness

,hospital_id,heart_rate_median,sbp_median,map_median,dbp_median,resp_rate_median,spont_resp_rate_median,core_temp_median,spo2_median,sao2_median,scvo2_median,cvp_median,fio2_median,feo2_median,peep_median,delta_p_median,insp_pressure_median,compliance_median,ie_ratio_median,vt_median,vt_per_kg_ibw_median,etco2_median,pf_ratio_median,pao2_median,paco2_median,ph_art_median,bicarbonate_art_median,base_excess_art_median,lactate_art_median,pap_sys_median,pap_mean_median,pap_dias_median,pcwp_median,cardiac_output_bolus_median,cardiac_output_cont_median,cardiac_index_bolus_median,cardiac_index_cont_median,stroke_volume_bolus_median,stroke_volume_cont_median,stroke_index_bolus_median,stroke_index_cont_median,svri_median,pvri_median,gedvi_median,evlwi_median,hemoglobin_median,hematocrit_median,wbc_median,platelets_median,lymph_abs_median,lymph_pct_median,inr_median,ptt_median,d_dimer_median,albumin_median,crp_median,pct_median,il6_median,bilirubin_total_median,urea_median,creatinine_median,bnp_median,ntprobnp_median,ast_median,alt_median,ldh_median,amylase_median,lipase_median,troponin_median,ck_median,ck_mb_median,fluid_balance_24h_median,position_therapy_median,ecmo_median,ecmo_o2_median,extracorp_blood_flow_median,extracorp_o2_flow_median,inhaled_no_median,inhaled_iloprost_median,dobutamine_iv_cont_median,epinephrine_iv_cont_median,norepinephrine_iv_cont_median,vasopressin_iv_cont_median,milrinone_iv_cont_median,levosimendan_iv_cont_median,terlipressin_iv_bolus_median,propofol_iv_cont_median,midazolam_iv_cont_median,clonidine_iv_cont_median,dexmedetomidine_iv_cont_median,ketanest_iv_cont_median,isoflurane_inh_median,sevoflurane_inh_median,sufentanil_iv_cont_median,fentanyl_iv_cont_median,morphine_iv_cont_median,rocuronium_iv_bolus_median,furosemide_iv_cont_median,hydrocortisone_iv_bolus_median,prednisolone_iv_bolus_median,dexamethasone_iv_bolus_median,fludrocortisone_po_bolus_median,sofa_median
0,asic_UK02,0.00000,0.069519,0.069519,0.069519,0.005348,1.000000,1.000000,0.005348,1.000000,0.010695,1.000000,0.168449,1.0,0.189840,1.000000,0.184492,0.179144,0.556150,1.000000,0.181818,0.181818,0.208556,0.037433,0.037433,1.000000,0.034759,0.034759,0.034759,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.0,1.000000,1.0,1.000000,1.000000,1.0,1.000000,1.000000,0.529412,0.534759,0.524064,0.534759,1.000000,1.000000,0.366310,0.286096,1.000000,0.652406,0.644385,1.000000,1.000000,1.000000,0.644385,0.652406,1.000000,1.000000,0.612299,1.000000,1.000000,1.000000,0.647059,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.0,1.000000,1.000000,0.323529,1.000000,1.000000,1.000000,1.0,0.719251,1.000000,1.00000,1.000000,1.000000,1.000000,1.0,0.379679,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.0
1,asic_UK04,0.00000,0.018051,0.018051,0.018051,0.021661,0.169675,0.036101,0.000000,0.043321,1.000000,0.985560,0.166065,1.0,0.202166,0.537906,0.166065,0.270758,0.638989,1.000000,0.202166,0.425993,0.256318,0.039711,0.039711,0.039711,0.039711,0.039711,0.039711,1.000000,1.000000,1.000000,1.000000,0.971119,1.0,0.974729,0.956679,1.0,0.953069,1.0,0.960289,0.960289,1.0,0.956679,0.956679,0.606498,0.606498,0.599278,0.606498,0.981949,0.978339,0.592058,0.584838,1.000000,0.960289,0.638989,0.895307,0.913357,0.866426,0.815884,0.808664,1.000000,1.000000,0.844765,0.841155,0.620939,0.996390,0.989170,0.935018,0.797834,0.924188,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.00000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.0,1.000000,1.0,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,1.0
2,asic_UK07,0.00000,0.000000,0.000000,0.000000,0.167254,0.165493,0.045775,0.000000,0.003521,0.945423,0.362676,0.140845,1.0,0.165493,0.165493,0.165493,1.000000,0.526408,0.170775,0.165493,0.235915,0.660211,0.667254,0.003521,0.003521,0.003521,0.003521,0.014085,0.915493,0.919014,0.920775,0.992958,1.000000,1.0,0.825704,0.772887,1.0,1.000000,1.0,1.000000,0.7

## Working Objects

The main notebook variables are:

- `harmonized_dataset`: harmonized static/dynamic tables plus stay-ID QC
- `final_static`: harmonized static table
- `final_dynamic`: harmonized dynamic table
- `chapter1_dataset`: Chapter 1 cohort plus 8-hour block outputs
- `final_cohort`: authoritative stay-level cohort table
- `chapter1_stays`: retained Chapter 1 stay-level cohort
- `chapter1_block_index`: completed 8-hour block index
- `chapter1_blocked_dynamic_features`: block-level aggregated dynamic feature table
- `harmonized_output_paths`: written static/dynamic/qc artifact paths
- `chapter1_output_paths`: written cohort/block artifact paths
